# 18 - Train Sevilla Dish NER Model v1

Este notebook entrena el primer modelo NER real de Hidden Gems para detectar menciones de platos en reseñas gastronómicas de Sevilla.

El objetivo es sustituir o reforzar la detección híbrida actual basada en reglas/diccionario mediante un modelo de `Token Classification`.

Flujo:

```text
reviews Sevilla
→ entidades DISH anotadas/preanotadas
→ etiquetas BIO
→ fine-tuning BETO
→ evaluación
→ export del modelo

```

Modelo base:

dccuchile/bert-base-spanish-wwm-cased

Etiquetas:

O
B-DISH
I-DISH

Este modelo será usado posteriormente para generar nuevas menciones de platos sobre las reviews de Sevilla.

In [1]:
# ============================================================
# 01. Instalación e imports
# ============================================================

!pip -q install -U transformers datasets evaluate seqeval accelerate

import os
import re
import json
import math
import random
import shutil
import zipfile
from pathlib import Path
from datetime import datetime, timezone
from typing import Any, Dict, List, Tuple, Optional

import numpy as np
import pandas as pd
import torch

from sklearn.model_selection import train_test_split

from datasets import Dataset, DatasetDict
import evaluate

from transformers import (
    AutoTokenizer,
    AutoConfig,
    AutoModelForTokenClassification,
    DataCollatorForTokenClassification,
    Trainer,
    TrainingArguments,
    EarlyStoppingCallback,
    set_seed,
)

os.environ["TOKENIZERS_PARALLELISM"] = "false"

print("Torch:", torch.__version__)
print("CUDA available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 2.3 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.6/10.6 MB 90.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 529.0/529.0 kB 25.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 5.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 383.7/383.7 kB 17.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 661.5/661.5 kB 37.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.5/4.5 MB 96.9 MB/s eta 0:00:00:00:01
Torch: 2.10.0+cu128
CUDA available: True
GPU: Tesla T4


In [2]:
# ============================================================
# 02. Configuración general - NER v1.2
# ============================================================

SEED = 42
set_seed(SEED)
random.seed(SEED)
np.random.seed(SEED)

NOTEBOOK_NAME = "18_train_sevilla_dish_ner_model_v1_2"
VERSION = "sevilla_dish_ner_beto_v1_2"

MODEL_CHECKPOINT = "dccuchile/bert-base-spanish-wwm-cased"

OUTPUT_ROOT = Path("/kaggle/working")
MODEL_OUTPUT_DIR = OUTPUT_ROOT / VERSION
REPORTS_DIR = OUTPUT_ROOT / "reports"
PREDICTIONS_DIR = OUTPUT_ROOT / "predictions"

for path in [MODEL_OUTPUT_DIR, REPORTS_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

LABEL_LIST = ["O", "B-DISH", "I-DISH"]
LABEL2ID = {label: idx for idx, label in enumerate(LABEL_LIST)}
ID2LABEL = {idx: label for label, idx in LABEL2ID.items()}

MAX_LENGTH = 384

TRAIN_BATCH_SIZE = 8
EVAL_BATCH_SIZE = 8
GRADIENT_ACCUMULATION_STEPS = 2

# Single-stage training: volvemos a la filosofía de v1, pero más calibrada.
NUM_EPOCHS = 6
LEARNING_RATE = 1.8e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.08

USE_FP16 = bool(torch.cuda.is_available())

# Pesos conservadores:
# No queremos sobreincentivar B/I-DISH porque la v1 ya tenía buen recall
# y el problema principal era precision/spans.
MANUAL_LABEL_WEIGHTS = {
    "O": 0.95,
    "B-DISH": 1.15,
    "I-DISH": 1.10,
}

# Suavizado leve para evitar sobreconfianza.
LABEL_SMOOTHING = 0.02

# El checkpoint se seleccionará por F0.5, que premia más precision que recall.
BEST_MODEL_METRIC = "f0_5"

CONFIG = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "model_checkpoint": MODEL_CHECKPOINT,
    "labels": LABEL_LIST,
    "max_length": MAX_LENGTH,
    "train_batch_size": TRAIN_BATCH_SIZE,
    "eval_batch_size": EVAL_BATCH_SIZE,
    "gradient_accumulation_steps": GRADIENT_ACCUMULATION_STEPS,
    "epochs": NUM_EPOCHS,
    "learning_rate": LEARNING_RATE,
    "weight_decay": WEIGHT_DECAY,
    "warmup_ratio": WARMUP_RATIO,
    "manual_label_weights": MANUAL_LABEL_WEIGHTS,
    "label_smoothing": LABEL_SMOOTHING,
    "best_model_metric": BEST_MODEL_METRIC,
    "seed": SEED,
    "fp16": USE_FP16,
}

print(json.dumps(CONFIG, indent=2, ensure_ascii=False))

{
  "notebook": "18_train_sevilla_dish_ner_model_v1_2",
  "version": "sevilla_dish_ner_beto_v1_2",
  "model_checkpoint": "dccuchile/bert-base-spanish-wwm-cased",
  "labels": [
    "O",
    "B-DISH",
    "I-DISH"
  ],
  "max_length": 384,
  "train_batch_size": 8,
  "eval_batch_size": 8,
  "gradient_accumulation_steps": 2,
  "epochs": 6,
  "learning_rate": 1.8e-05,
  "weight_decay": 0.01,
  "warmup_ratio": 0.08,
  "manual_label_weights": {
    "O": 0.95,
    "B-DISH": 1.15,
    "I-DISH": 1.1
  },
  "label_smoothing": 0.02,
  "best_model_metric": "f0_5",
  "seed": 42,
  "fp16": true
}


In [3]:
# ============================================================
# 03. Localización automática del dataset
# ============================================================

def find_input_file(filename_patterns: List[str]) -> Path:
    search_roots = [
        Path("/kaggle/input"),
        Path("/kaggle/working"),
    ]

    candidates = []
    for root in search_roots:
        if root.exists():
            for pattern in filename_patterns:
                candidates.extend(root.rglob(pattern))

    if not candidates:
        raise FileNotFoundError(
            f"No se encontró ningún archivo con patrones: {filename_patterns}. "
            f"Revisa que hayas añadido el dataset privado de Kaggle."
        )

    candidates = sorted(candidates, key=lambda p: len(str(p)))
    return candidates[0]


DATA_PATH = find_input_file([
    "sevilla_ner_annotation_dataset_v1_extended.jsonl",
    "sevilla_ner_annotation_dataset_v1_extended.csv",
])

print("Dataset encontrado:", DATA_PATH)

Dataset encontrado: /kaggle/input/datasets/ivanarteagacordero/hidden-gems-sevilla-v1/sevilla_ner_annotation_dataset_v1_extended.csv


In [4]:
# ============================================================
# 04. Carga del dataset
# ============================================================

def read_jsonl(path: Path) -> pd.DataFrame:
    rows = []
    with path.open("r", encoding="utf-8") as f:
        for line in f:
            line = line.strip()
            if line:
                rows.append(json.loads(line))
    return pd.DataFrame(rows)


if DATA_PATH.suffix.lower() == ".jsonl":
    df = read_jsonl(DATA_PATH)
else:
    df = pd.read_csv(DATA_PATH)

print("Shape:", df.shape)
display(df.head(3))
print(df.columns.tolist())

required_cols = [
    "document_id",
    "review_id",
    "text",
    "preannotated_entities_json",
    "manual_entities_json",
    "annotation_status",
]

missing = [c for c in required_cols if c not in df.columns]
if missing:
    raise KeyError(f"Faltan columnas obligatorias: {missing}")

df["text"] = df["text"].fillna("").astype(str)
df["annotation_status"] = df["annotation_status"].fillna("unknown").astype(str)

print("Distribución annotation_status:")
display(df["annotation_status"].value_counts(dropna=False))

Shape: (5130, 22)


,document_id,review_id,text,text_length_chars,preannotation_count,preannotated_entities_json,manual_entities_json,annotation_status,place_id,place_name,...,longitude,neighborhood_id,neighborhood_name,district_id,district_name,rating_value,review_language,review_created_at,source_review_id,source_place_record_id
0,sevilla_google_review_0d0946cb-a72c-4197-9d43-...,0d0946cb-a72c-4197-9d43-b5f3dbfcc620,Sin dudas el mejor Açai que he probado aqui en...,221,2,"[{""start"": 19, ""end"": 23, ""label"": ""DISH"", ""te...",[],pending,8cb19db3-e9a8-4a42-be63-73e86e22a3d9,Tropitali,...,-5.985623,e554ef0e-d9f1-4d89-8777-f92cee028f7f,SAN JULIAN,5bd4fbf1-457b-4dcb-91f1-089ac449a5db,Casco Antiguo,5.0,es,2026-05-10T19:07:36.701337+02:00,a299c238d5349ed3eb5fc64b5db7e6def4db213c4ed817...,ChIJ50jgAn5tEg0RBySkaJ7I8eE
1,sevilla_google_review_b680ad6e-8999-48a0-a754-...,b680ad6e-8999-48a0-a754-84ea920429ab,Buena comida y con un excelente servicio de Le...,124,0,[],[],pending,a6ba370c-eb3d-4456-94cf-58617d1a3b0a,Cariña VZ,...,-5.977433,65af0982-2885-49df-9f09-8b784c2f3d7b,GIRALDA SUR,22b433e4-3b14-4b09-bfaa-f0ca1d7e6bdc,Sur,5.0,es,2026-05-10T18:17:25.948671+02:00,4897b8c24f949dda32b4e6c5cad74f4a91228c64182c5c...,ChIJuQj13EFvEg0RPkUdEUzL0BA
2,sevilla_google_review_b2ac02e7-4348-481d-ad2e-...,b2ac02e7-4348-481d-ad2e-cb37ab9c2685,Demasiada tiempo de espera para recibir mi ped...,106,0,[],[],pending,7d39aad6-1d3f-4fc8-8d4b-22896354507e,TOTORA,...,-5.951597,76a474aa-a390-48b4-a7ad-d68f6d747832,LA PLATA,09b2875d-a826-4c21-bd78-1d8fa48009de,Cerro - Amate,2.0,es,2026-05-10T18:03:34.257906+02:00,c19ee9e9030a22bbcd80d00ef0680365a36e284ccb7295...,ChIJmU6FwYdvEg0RYLF2daPZQkc


['document_id', 'review_id', 'text', 'text_length_chars', 'preannotation_count', 'preannotated_entities_json', 'manual_entities_json', 'annotation_status', 'place_id', 'place_name', 'address_text', 'latitude', 'longitude', 'neighborhood_id', 'neighborhood_name', 'district_id', 'district_name', 'rating_value', 'review_language', 'review_created_at', 'source_review_id', 'source_place_record_id']
Distribución annotation_status:


annotation_status
pending                              4110
manual_augmented_multi_dish           490
manual_augmented_single_dish          420
manual_augmented_negative_control     100
manual_augmented_hard_case             10
Name: count, dtype: int64

In [5]:
# ============================================================
# 05. Parseo de entidades y selección de etiquetas
# ============================================================

def parse_entities(value: Any) -> List[Dict[str, Any]]:
    if value is None:
        return []

    if isinstance(value, float) and math.isnan(value):
        return []

    if isinstance(value, list):
        raw = value
    else:
        text_value = str(value).strip()
        if not text_value or text_value.lower() in ["nan", "none", "null"]:
            return []
        try:
            raw = json.loads(text_value)
        except Exception:
            return []

    entities = []
    for ent in raw:
        try:
            start = int(ent.get("start"))
            end = int(ent.get("end"))
            label = str(ent.get("label", "DISH"))
            ent_text = str(ent.get("text", ""))
            if label != "DISH":
                continue
            if start < 0 or end <= start:
                continue
            entities.append({
                "start": start,
                "end": end,
                "label": "DISH",
                "text": ent_text,
                "source": ent.get("source", "unknown"),
            })
        except Exception:
            continue

    entities = sorted(entities, key=lambda x: (x["start"], x["end"]))
    return entities


def choose_entities(row: pd.Series) -> Tuple[List[Dict[str, Any]], str]:
    manual_entities = parse_entities(row.get("manual_entities_json"))
    pre_entities = parse_entities(row.get("preannotated_entities_json"))

    status = str(row.get("annotation_status", ""))

    # Los casos manual_augmented tienen etiquetas gold manuales.
    if status.startswith("manual_augmented"):
        return manual_entities, "manual_gold"

    # Las filas reales pendientes usan preanotaciones débiles del sistema híbrido.
    if pre_entities:
        return pre_entities, "weak_hybrid"

    # Filas sin platos detectados o controles negativos.
    return [], "negative_or_unlabeled"


df["entities"] = None
df["label_source"] = None

entities_col = []
source_col = []

for _, row in df.iterrows():
    ents, source = choose_entities(row)
    entities_col.append(ents)
    source_col.append(source)

df["entities"] = entities_col
df["label_source"] = source_col
df["entity_count"] = df["entities"].map(len)

print("Distribución label_source:")
display(df["label_source"].value_counts())

print("Distribución entity_count:")
display(df["entity_count"].value_counts().sort_index().head(20))

display(df[["document_id", "annotation_status", "label_source", "entity_count", "text", "entities"]].head(10))

Distribución label_source:


label_source
negative_or_unlabeled    2658
weak_hybrid              1452
manual_gold              1020
Name: count, dtype: int64

Distribución entity_count:


entity_count
0    2758
1    1115
2     736
3     300
4     129
5      39
6      28
7      15
8       7
9       3
Name: count, dtype: int64

,document_id,annotation_status,label_source,entity_count,text,entities
0,sevilla_google_review_0d0946cb-a72c-4197-9d43-...,pending,weak_hybrid,2,Sin dudas el mejor Açai que he probado aqui en...,"[{'start': 19, 'end': 23, 'label': 'DISH', 'te..."
1,sevilla_google_review_b680ad6e-8999-48a0-a754-...,pending,negative_or_unlabeled,0,Buena comida y con un excelente servicio de Le...,[]
2,sevilla_google_review_b2ac02e7-4348-481d-ad2e-...,pending,negative_or_unlabeled,0,Demasiada tiempo de espera para recibir mi ped...,[]
3,sevilla_google_review_80af93bc-d556-46a4-8254-...,pending,negative_or_unlabeled,0,"Una decepción, esperé 2 horas mi pedido, la ca...",[]
4,sevilla_google_review_227b359d-b549-4d0c-957e-...,pending,negative_or_unlabeled,0,la comida peruana en sevilla anticuchos de pr...,[]
5,sevilla_google_review_e8f1661c-93ab-4fdc-bb27-...,pending,negative_or_unlabeled,0,Preguntamos por el precio de la botella de vin...,[]
6,sevilla_google_review_2b16e153-117a-441d-af78-...,pending,negative_or_unlabeled,0,"La comida estaba bastante buena, pero en cuant...",[]
7,sevilla_google_review_944c8719-53fd-49a0-9da4-...,pending,weak_hybrid,1,Visité este restaurante venezolano y fue una e...,"[{'start': 102, 'end': 107, 'label': 'DISH', '..."
8,sevilla_google_review_7b47e63c-8f6c-4fe3-85f1-...,pending,weak_hybrid,1,"El mejor lugar de comida venezolana, llevo año...","[{'start': 144, 'end': 153, 'label': 'DISH', '..."
9,sevilla_google_review_6ae9c075-795e-4f4a-9d8c-...,pending,negative_or_unlabeled,0,Nos atendió Alba y ella sola hace que la exper...,[]


In [6]:
# ============================================================
# 06. Validación de spans
# ============================================================

def validate_entity_spans(row: pd.Series) -> List[Dict[str, Any]]:
    text = row["text"]
    issues = []

    for ent in row["entities"]:
        start, end = ent["start"], ent["end"]
        expected = ent.get("text", "")
        actual = text[start:end]

        if start < 0 or end > len(text) or end <= start:
            issues.append({
                "document_id": row["document_id"],
                "issue": "out_of_bounds",
                "start": start,
                "end": end,
                "expected": expected,
                "actual": actual,
            })
        elif expected and actual.lower() != expected.lower():
            issues.append({
                "document_id": row["document_id"],
                "issue": "text_mismatch",
                "start": start,
                "end": end,
                "expected": expected,
                "actual": actual,
            })

    return issues


span_issues = []
for _, row in df.iterrows():
    span_issues.extend(validate_entity_spans(row))

span_issues_df = pd.DataFrame(span_issues)
print("Span issues:", len(span_issues_df))

if len(span_issues_df):
    display(span_issues_df.head(20))

# Para entrenamiento, descartamos solo entidades fuera de límites.
def clean_entities(row: pd.Series) -> List[Dict[str, Any]]:
    text = row["text"]
    clean = []
    for ent in row["entities"]:
        start, end = ent["start"], ent["end"]
        if 0 <= start < end <= len(text):
            clean.append(ent)
    return clean

df["entities"] = df.apply(clean_entities, axis=1)
df["entity_count"] = df["entities"].map(len)

print("Filas:", len(df))
print("Entidades totales:", int(df["entity_count"].sum()))

Span issues: 0
Filas: 5130
Entidades totales: 4554


In [7]:
# ============================================================
# 07. Preparación de splits train/validation/test
# ============================================================

# Creamos una columna de estratificación sencilla.
df["has_entity"] = df["entity_count"] > 0
df["split_group"] = (
    df["label_source"].astype(str)
    + "__"
    + df["has_entity"].astype(str)
    + "__"
    + df["annotation_status"].astype(str)
)

# Para evitar grupos demasiado pequeños en stratify.
group_counts = df["split_group"].value_counts()
rare_groups = set(group_counts[group_counts < 3].index)
df["split_group_safe"] = df["split_group"].where(~df["split_group"].isin(rare_groups), "rare")

train_df, temp_df = train_test_split(
    df,
    test_size=0.20,
    random_state=SEED,
    stratify=df["split_group_safe"],
)

temp_group_counts = temp_df["split_group_safe"].value_counts()
if (temp_group_counts < 2).any():
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=SEED,
    )
else:
    val_df, test_df = train_test_split(
        temp_df,
        test_size=0.50,
        random_state=SEED,
        stratify=temp_df["split_group_safe"],
    )

train_df = train_df.reset_index(drop=True)
val_df = val_df.reset_index(drop=True)
test_df = test_df.reset_index(drop=True)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Test:", test_df.shape)

print("\nTrain label_source:")
display(train_df["label_source"].value_counts())

print("\nValidation label_source:")
display(val_df["label_source"].value_counts())

print("\nTest label_source:")
display(test_df["label_source"].value_counts())

split_summary = {
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(val_df)),
    "test_rows": int(len(test_df)),
    "train_entities": int(train_df["entity_count"].sum()),
    "validation_entities": int(val_df["entity_count"].sum()),
    "test_entities": int(test_df["entity_count"].sum()),
}

print(json.dumps(split_summary, indent=2, ensure_ascii=False))

Train: (4104, 28)
Validation: (513, 28)
Test: (513, 28)

Train label_source:


label_source
negative_or_unlabeled    2126
weak_hybrid              1162
manual_gold               816
Name: count, dtype: int64


Validation label_source:


label_source
negative_or_unlabeled    266
weak_hybrid              145
manual_gold              102
Name: count, dtype: int64


Test label_source:


label_source
negative_or_unlabeled    266
weak_hybrid              145
manual_gold              102
Name: count, dtype: int64

{
  "train_rows": 4104,
  "validation_rows": 513,
  "test_rows": 513,
  "train_entities": 3607,
  "validation_entities": 489,
  "test_entities": 458
}


In [8]:
# ============================================================
# 07B. Dataset específico para fine-tuning de precisión
# ============================================================

# Para la segunda fase usamos principalmente ejemplos manuales/aumentados.
# Evitamos entrenar demasiado sobre negativos reales no revisados, porque pueden contener platos no anotados.

stage2_mask = (
    train_df["annotation_status"].astype(str).str.startswith("manual_augmented")
)

stage2_train_df = train_df[stage2_mask].copy().reset_index(drop=True)

if len(stage2_train_df) == 0:
    raise ValueError("No hay filas manual_augmented para stage2. Revisa el dataset extendido.")

print("Stage2 train rows:", len(stage2_train_df))
print("Stage2 entities:", int(stage2_train_df["entity_count"].sum()))
display(stage2_train_df["annotation_status"].value_counts())

Stage2 train rows: 816
Stage2 entities: 1262


annotation_status
manual_augmented_multi_dish          392
manual_augmented_single_dish         336
manual_augmented_negative_control     80
manual_augmented_hard_case             8
Name: count, dtype: int64

In [9]:
# ============================================================
# 08. Conversión a Hugging Face Dataset
# ============================================================

def dataframe_to_records(dataframe: pd.DataFrame) -> List[Dict[str, Any]]:
    cols = [
        "document_id",
        "review_id",
        "text",
        "entities",
        "annotation_status",
        "label_source",
        "entity_count",
        "place_name",
        "district_name",
        "neighborhood_name",
        "rating_value",
        "review_language",
    ]

    available_cols = [c for c in cols if c in dataframe.columns]
    records = []

    for _, row in dataframe[available_cols].iterrows():
        record = row.to_dict()
        for k, v in list(record.items()):
            if isinstance(v, float) and math.isnan(v):
                record[k] = None
        records.append(record)

    return records


dataset = DatasetDict({
    "train": Dataset.from_list(dataframe_to_records(train_df)),
    "validation": Dataset.from_list(dataframe_to_records(val_df)),
    "test": Dataset.from_list(dataframe_to_records(test_df)),
    "stage2_train": Dataset.from_list(dataframe_to_records(stage2_train_df)),
})

dataset

DatasetDict({
    train: Dataset({
        features: ['document_id', 'review_id', 'text', 'entities', 'annotation_status', 'label_source', 'entity_count', 'place_name', 'district_name', 'neighborhood_name', 'rating_value', 'review_language'],
        num_rows: 4104
    })
    validation: Dataset({
        features: ['document_id', 'review_id', 'text', 'entities', 'annotation_status', 'label_source', 'entity_count', 'place_name', 'district_name', 'neighborhood_name', 'rating_value', 'review_language'],
        num_rows: 513
    })
    test: Dataset({
        features: ['document_id', 'review_id', 'text', 'entities', 'annotation_status', 'label_source', 'entity_count', 'place_name', 'district_name', 'neighborhood_name', 'rating_value', 'review_language'],
        num_rows: 513
    })
    stage2_train: Dataset({
        features: ['document_id', 'review_id', 'text', 'entities', 'annotation_status', 'label_source', 'entity_count', 'place_name', 'district_name', 'neighborhood_name', 'rating

In [10]:
# ============================================================
# 09. Tokenizador y alineación BIO
# ============================================================

tokenizer = AutoTokenizer.from_pretrained(MODEL_CHECKPOINT, use_fast=True)

def entities_to_bio_labels(
    text: str,
    entities: List[Dict[str, Any]],
    offsets: List[Tuple[int, int]],
) -> List[int]:
    labels = []

    sorted_entities = sorted(entities, key=lambda e: (int(e["start"]), int(e["end"])))
    previous_entity_idx = None

    for token_start, token_end in offsets:
        if token_start == token_end:
            labels.append(-100)
            previous_entity_idx = None
            continue

        matched_idx = None
        for idx, ent in enumerate(sorted_entities):
            ent_start = int(ent["start"])
            ent_end = int(ent["end"])

            if token_start < ent_end and token_end > ent_start:
                matched_idx = idx
                break

        if matched_idx is None:
            labels.append(LABEL2ID["O"])
            previous_entity_idx = None
        else:
            ent = sorted_entities[matched_idx]
            ent_start = int(ent["start"])

            if matched_idx != previous_entity_idx or token_start <= ent_start < token_end:
                labels.append(LABEL2ID["B-DISH"])
            else:
                labels.append(LABEL2ID["I-DISH"])

            previous_entity_idx = matched_idx

    return labels


def tokenize_and_align_labels(batch: Dict[str, List[Any]]) -> Dict[str, Any]:
    tokenized = tokenizer(
        batch["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
        return_offsets_mapping=True,
    )

    all_labels = []

    for i, offsets in enumerate(tokenized["offset_mapping"]):
        text = batch["text"][i]
        entities = batch["entities"][i]
        labels = entities_to_bio_labels(text, entities, offsets)
        all_labels.append(labels)

    tokenized["labels"] = all_labels
    tokenized.pop("offset_mapping")

    return tokenized


tokenized_dataset = dataset.map(
    tokenize_and_align_labels,
    batched=True,
    remove_columns=dataset["train"].column_names,
)

tokenized_dataset

config.json:   0%|          | 0.00/648 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/364 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/134 [00:00<?, ?B/s]

Map:   0%|          | 0/4104 [00:00<?, ? examples/s]

Map:   0%|          | 0/513 [00:00<?, ? examples/s]

Map:   0%|          | 0/513 [00:00<?, ? examples/s]

Map:   0%|          | 0/816 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 4104
    })
    validation: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 513
    })
    test: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 513
    })
    stage2_train: Dataset({
        features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
        num_rows: 816
    })
})

In [11]:
# ============================================================
# 10. Comprobación visual de alineación
# ============================================================

def inspect_tokenized_example(split: str = "train", index: int = 0):
    original = dataset[split][index]
    encoded = tokenizer(
        original["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        return_offsets_mapping=True,
    )

    labels = entities_to_bio_labels(
        original["text"],
        original["entities"],
        encoded["offset_mapping"],
    )

    tokens = tokenizer.convert_ids_to_tokens(encoded["input_ids"])

    rows = []
    for tok, offset, label_id in zip(tokens, encoded["offset_mapping"], labels):
        label = "IGN" if label_id == -100 else ID2LABEL[label_id]
        rows.append({
            "token": tok,
            "offset": offset,
            "label": label,
        })

    print("TEXT:")
    print(original["text"])
    print("\nENTITIES:")
    print(original["entities"])

    return pd.DataFrame(rows)


# Buscar un ejemplo con entidades.
example_indices = [i for i, ex in enumerate(dataset["train"]) if ex["entity_count"] > 0]
display(inspect_tokenized_example("train", example_indices[0]).head(80))

TEXT:
Compartimos gazpacho andaluz y ramen de miso; el primero estaba espectacular y el segundo correcto. Volveríamos para comparar con otros platos.

ENTITIES:
[{'end': 28, 'label': 'DISH', 'source': 'manual_augmented_gold_v1', 'start': 12, 'text': 'gazpacho andaluz'}, {'end': 44, 'label': 'DISH', 'source': 'manual_augmented_gold_v1', 'start': 31, 'text': 'ramen de miso'}]


,token,offset,label
0,[CLS],"(0, 0)",IGN
1,Compar,"(0, 6)",O
2,##timos,"(6, 11)",O
3,ga,"(12, 14)",B-DISH
4,##z,"(14, 15)",I-DISH
5,##pac,"(15, 18)",I-DISH
6,##ho,"(18, 20)",I-DISH
7,anda,"(21, 25)",I-DISH
8,##lu,"(25, 27)",I-DISH
9,##z,"(27, 28)",I-DISH


In [12]:
# ============================================================
# 11. Métricas NER con F0.5
# ============================================================

seqeval = evaluate.load("seqeval")

def logits_to_label_sequences(predictions, labels):
    preds = np.argmax(predictions, axis=-1)

    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(preds, labels):
        pred_labels = []
        gold_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue

            pred_labels.append(ID2LABEL[int(pred_id)])
            gold_labels.append(ID2LABEL[int(label_id)])

        true_predictions.append(pred_labels)
        true_labels.append(gold_labels)

    return true_predictions, true_labels


def compute_fbeta(precision: float, recall: float, beta: float = 0.5) -> float:
    if precision + recall == 0:
        return 0.0
    return (1 + beta**2) * precision * recall / ((beta**2 * precision) + recall)


def compute_metrics(eval_pred):
    predictions, labels = eval_pred
    true_predictions, true_labels = logits_to_label_sequences(predictions, labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
        zero_division=0,
    )

    precision = results.get("overall_precision", 0.0)
    recall = results.get("overall_recall", 0.0)
    f1 = results.get("overall_f1", 0.0)
    accuracy = results.get("overall_accuracy", 0.0)
    f0_5 = compute_fbeta(precision, recall, beta=0.5)

    overall = {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f0_5": f0_5,
        "accuracy": accuracy,
    }

    if "DISH" in results:
        dish_precision = results["DISH"].get("precision", 0.0)
        dish_recall = results["DISH"].get("recall", 0.0)
        dish_f1 = results["DISH"].get("f1", 0.0)
        overall["dish_precision"] = dish_precision
        overall["dish_recall"] = dish_recall
        overall["dish_f1"] = dish_f1
        overall["dish_f0_5"] = compute_fbeta(dish_precision, dish_recall, beta=0.5)
        overall["dish_number"] = results["DISH"].get("number", 0)

    return overall

In [13]:
# ============================================================
# 12. Pesos de clases conservadores
# ============================================================

def inspect_label_counts(tokenized_split: Dataset) -> Dict[str, int]:
    counts = np.zeros(len(LABEL_LIST), dtype=np.int64)

    for row in tokenized_split:
        for label_id in row["labels"]:
            if label_id != -100:
                counts[int(label_id)] += 1

    out = {}
    print("Label counts:")
    for idx, count in enumerate(counts):
        out[ID2LABEL[idx]] = int(count)
        print(ID2LABEL[idx], int(count))

    return out


label_counts = inspect_label_counts(tokenized_dataset["train"])

LABEL_WEIGHTS = torch.tensor(
    [
        MANUAL_LABEL_WEIGHTS["O"],
        MANUAL_LABEL_WEIGHTS["B-DISH"],
        MANUAL_LABEL_WEIGHTS["I-DISH"],
    ],
    dtype=torch.float,
)

print("\nManual label weights:")
for idx, weight in enumerate(LABEL_WEIGHTS.tolist()):
    print(ID2LABEL[idx], round(float(weight), 4))

Label counts:
O 270395
B-DISH 3603
I-DISH 7970

Manual label weights:
O 0.95
B-DISH 1.15
I-DISH 1.1


In [14]:
# ============================================================
# 13. Trainer con loss ponderada y label smoothing
# ============================================================

class WeightedTokenClassificationTrainer(Trainer):
    def __init__(
        self,
        *args,
        label_weights: Optional[torch.Tensor] = None,
        label_smoothing: float = 0.0,
        **kwargs,
    ):
        super().__init__(*args, **kwargs)
        self.label_weights = label_weights
        self.label_smoothing = label_smoothing

    def compute_loss(self, model, inputs, return_outputs=False, num_items_in_batch=None):
        labels = inputs.get("labels")
        outputs = model(**inputs)
        logits = outputs.get("logits")

        num_labels = logits.shape[-1]
        weights = self.label_weights.to(logits.device) if self.label_weights is not None else None

        loss_fct = torch.nn.CrossEntropyLoss(
            weight=weights,
            ignore_index=-100,
            label_smoothing=self.label_smoothing,
        )

        loss = loss_fct(
            logits.reshape(-1, num_labels),
            labels.reshape(-1),
        )

        return (loss, outputs) if return_outputs else loss

In [15]:
# ============================================================
# 14. Modelo y argumentos de entrenamiento v1.2
# ============================================================

model_config = AutoConfig.from_pretrained(
    MODEL_CHECKPOINT,
    num_labels=len(LABEL_LIST),
    id2label=ID2LABEL,
    label2id=LABEL2ID,
)

model = AutoModelForTokenClassification.from_pretrained(
    MODEL_CHECKPOINT,
    config=model_config,
)

data_collator = DataCollatorForTokenClassification(tokenizer=tokenizer)

def compute_warmup_steps(train_size: int, epochs: int) -> int:
    steps_per_epoch = math.ceil(
        train_size / (TRAIN_BATCH_SIZE * GRADIENT_ACCUMULATION_STEPS)
    )
    total_steps = steps_per_epoch * epochs
    return int(total_steps * WARMUP_RATIO)


WARMUP_STEPS = compute_warmup_steps(
    train_size=len(tokenized_dataset["train"]),
    epochs=NUM_EPOCHS,
)

print("train_size:", len(tokenized_dataset["train"]))
print("warmup_steps:", WARMUP_STEPS)

def build_training_args():
    common_kwargs = dict(
        output_dir=str(MODEL_OUTPUT_DIR),
        learning_rate=LEARNING_RATE,
        per_device_train_batch_size=TRAIN_BATCH_SIZE,
        per_device_eval_batch_size=EVAL_BATCH_SIZE,
        gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
        num_train_epochs=NUM_EPOCHS,
        weight_decay=WEIGHT_DECAY,
        warmup_steps=WARMUP_STEPS,
        logging_steps=25,
        save_strategy="epoch",
        save_total_limit=2,
        load_best_model_at_end=True,
        metric_for_best_model=BEST_MODEL_METRIC,
        greater_is_better=True,
        fp16=USE_FP16,
        report_to="none",
        seed=SEED,
        remove_unused_columns=True,
    )

    try:
        return TrainingArguments(
            eval_strategy="epoch",
            **common_kwargs,
        )
    except TypeError:
        return TrainingArguments(
            evaluation_strategy="epoch",
            **common_kwargs,
        )

training_args = build_training_args()

trainer = WeightedTokenClassificationTrainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_dataset["train"],
    eval_dataset=tokenized_dataset["validation"],
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=2)],
    label_weights=LABEL_WEIGHTS,
    label_smoothing=LABEL_SMOOTHING,
)

print(training_args)

pytorch_model.bin:   0%|          | 0.00/440M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

[transformers] BertForTokenClassification LOAD REPORT from: dccuchile/bert-base-spanish-wwm-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.decoder.bias               | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING:	those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


train_size: 4104
warmup_steps: 123
TrainingArguments(
accelerator_config={'split_batches': False, 'dispatch_batches': None, 'even_batches': True, 'use_seedable_sampler': True, 'non_blocking': False, 'gradient_accumulation_kwargs': None, 'use_configured_state': False},
adam_beta1=0.9,
adam_beta2=0.999,
adam_epsilon=1e-08,
auto_find_batch_size=False,
average_tokens_across_devices=True,
batch_eval_metrics=False,
bf16=False,
bf16_full_eval=False,
data_seed=None,
dataloader_drop_last=False,
dataloader_num_workers=0,
dataloader_persistent_workers=False,
dataloader_pin_memory=True,
dataloader_prefetch_factor=None,
ddp_backend=None,
ddp_broadcast_buffers=None,
ddp_bucket_cap_mb=None,
ddp_find_unused_parameters=None,
ddp_static_graph=None,
ddp_timeout=1800,
debug=[],
deepspeed=None,
disable_tqdm=False,
do_eval=True,
do_predict=False,
do_train=False,
enable_jit_checkpoint=False,
eval_accumulation_steps=None,
eval_delay=0,
eval_do_concat_batches=True,
eval_on_start=False,
eval_steps=None,
eval_st

In [16]:
# ============================================================
# 15. Entrenamiento v1.2
# ============================================================

train_result = trainer.train()

train_metrics = train_result.metrics

trainer.save_model(str(MODEL_OUTPUT_DIR))
tokenizer.save_pretrained(str(MODEL_OUTPUT_DIR))

with (REPORTS_DIR / "train_metrics_v1_2.json").open("w", encoding="utf-8") as f:
    json.dump(train_metrics, f, indent=2, ensure_ascii=False)

print(json.dumps(train_metrics, indent=2, ensure_ascii=False))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,F0 5,Accuracy,Dish Precision,Dish Recall,Dish F1,Dish F0 5,Dish Number
1,0.267470,0.125481,0.655738,0.736196,0.693642,0.670391,0.987807,0.655738,0.736196,0.693642,0.670391,489
2,0.216127,0.112736,0.833002,0.856851,0.844758,0.837665,0.993235,0.833002,0.856851,0.844758,0.837665,489
3,0.211554,0.111660,0.801147,0.856851,0.828063,0.811701,0.993291,0.801147,0.856851,0.828063,0.811701,489
4,0.206805,0.110448,0.823985,0.871166,0.846918,0.833007,0.993764,0.823985,0.871166,0.846918,0.833007,489


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

{
  "train_runtime": 445.8369,
  "train_samples_per_second": 55.231,
  "train_steps_per_second": 1.736,
  "total_flos": 1811978741426688.0,
  "train_loss": 0.321223588876946,
  "epoch": 4.0
}


In [17]:
# ============================================================
# 16. Evaluación final base en validation y test
# ============================================================

validation_metrics = trainer.evaluate(tokenized_dataset["validation"])
test_metrics = trainer.evaluate(tokenized_dataset["test"])

metrics_report = {
    "config": CONFIG,
    "split_summary": split_summary,
    "train_metrics": train_metrics,
    "validation_metrics": validation_metrics,
    "test_metrics": test_metrics,
    "label_mapping": {
        "label2id": LABEL2ID,
        "id2label": ID2LABEL,
    },
}

with (REPORTS_DIR / "ner_metrics_v1_2_base.json").open("w", encoding="utf-8") as f:
    json.dump(metrics_report, f, indent=2, ensure_ascii=False)

print("Validation metrics:")
print(json.dumps(validation_metrics, indent=2, ensure_ascii=False))

print("\nTest metrics:")
print(json.dumps(test_metrics, indent=2, ensure_ascii=False))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,F0 5,Accuracy,Dish Precision,Dish Recall,Dish F1,Dish F0 5,Dish Number
0.206805,0.112736,4,0.833002,0.856851,0.844758,0.837665,0.993235,0.833002,0.856851,0.844758,0.837665,489


Training Loss,Validation Loss,Epoch,Precision,Recall,F1,F0 5,Accuracy,Dish Precision,Dish Recall,Dish F1,Dish F0 5,Dish Number
0.206805,0.112818,4,0.796218,0.827511,0.811563,0.802286,0.993104,0.796218,0.827511,0.811563,0.802286,458


Validation metrics:
{
  "eval_loss": 0.11273575574159622,
  "eval_precision": 0.8330019880715706,
  "eval_recall": 0.8568507157464212,
  "eval_f1": 0.844758064516129,
  "eval_f0_5": 0.8376649340263894,
  "eval_accuracy": 0.9932351549233039,
  "eval_dish_precision": 0.8330019880715706,
  "eval_dish_recall": 0.8568507157464212,
  "eval_dish_f1": 0.844758064516129,
  "eval_dish_f0_5": 0.8376649340263894,
  "eval_dish_number": 489
}

Test metrics:
{
  "eval_loss": 0.1128183975815773,
  "eval_precision": 0.7962184873949579,
  "eval_recall": 0.8275109170305677,
  "eval_f1": 0.8115631691648822,
  "eval_f0_5": 0.8022861981371718,
  "eval_accuracy": 0.9931036416136357,
  "eval_dish_precision": 0.7962184873949579,
  "eval_dish_recall": 0.8275109170305677,
  "eval_dish_f1": 0.8115631691648822,
  "eval_dish_f0_5": 0.8022861981371718,
  "eval_dish_number": 458
}


In [18]:
# ============================================================
# 16B. Calibración de threshold de entidad
# ============================================================

def softmax_np(logits: np.ndarray) -> np.ndarray:
    logits = logits - np.max(logits, axis=-1, keepdims=True)
    exp = np.exp(logits)
    return exp / np.sum(exp, axis=-1, keepdims=True)


def apply_entity_threshold(predictions: np.ndarray, labels: np.ndarray, threshold: float):
    probs = softmax_np(predictions)
    pred_ids = np.argmax(probs, axis=-1)

    adjusted = pred_ids.copy()

    for i in range(adjusted.shape[0]):
        for j in range(adjusted.shape[1]):
            if labels[i, j] == -100:
                continue

            pred_id = int(adjusted[i, j])

            if pred_id in [LABEL2ID["B-DISH"], LABEL2ID["I-DISH"]]:
                pred_prob = float(probs[i, j, pred_id])
                if pred_prob < threshold:
                    adjusted[i, j] = LABEL2ID["O"]

    return adjusted


def compute_seqeval_from_pred_ids(pred_ids: np.ndarray, labels: np.ndarray) -> Dict[str, float]:
    true_predictions = []
    true_labels = []

    for pred_seq, label_seq in zip(pred_ids, labels):
        pred_labels = []
        gold_labels = []

        for pred_id, label_id in zip(pred_seq, label_seq):
            if label_id == -100:
                continue

            pred_labels.append(ID2LABEL[int(pred_id)])
            gold_labels.append(ID2LABEL[int(label_id)])

        true_predictions.append(pred_labels)
        true_labels.append(gold_labels)

    results = seqeval.compute(
        predictions=true_predictions,
        references=true_labels,
        zero_division=0,
    )

    precision = results.get("overall_precision", 0.0)
    recall = results.get("overall_recall", 0.0)
    f1 = results.get("overall_f1", 0.0)

    return {
        "precision": precision,
        "recall": recall,
        "f1": f1,
        "f0_5": compute_fbeta(precision, recall, beta=0.5),
        "accuracy": results.get("overall_accuracy", 0.0),
    }


validation_pred_output = trainer.predict(tokenized_dataset["validation"])
val_logits = validation_pred_output.predictions
val_labels = validation_pred_output.label_ids

threshold_rows = []

# Probamos umbrales finos. 0.50-0.95 suele ser la zona útil.
for threshold in np.arange(0.40, 0.96, 0.025):
    adjusted_pred_ids = apply_entity_threshold(
        val_logits,
        val_labels,
        threshold=float(threshold),
    )
    metrics = compute_seqeval_from_pred_ids(adjusted_pred_ids, val_labels)
    metrics["threshold"] = round(float(threshold), 3)
    threshold_rows.append(metrics)

threshold_df = pd.DataFrame(threshold_rows)

# Elegimos por F0.5, pero exigimos recall mínimo razonable.
valid_threshold_df = threshold_df[threshold_df["recall"] >= 0.72].copy()

if len(valid_threshold_df):
    best_row = valid_threshold_df.sort_values(
        ["f0_5", "precision", "f1"],
        ascending=False,
    ).iloc[0]
else:
    best_row = threshold_df.sort_values(
        ["f0_5", "precision", "f1"],
        ascending=False,
    ).iloc[0]

BEST_ENTITY_THRESHOLD = float(best_row["threshold"])

threshold_path = REPORTS_DIR / "ner_threshold_calibration_v1_2.csv"
threshold_df.to_csv(threshold_path, index=False, encoding="utf-8")

print("Best threshold:", BEST_ENTITY_THRESHOLD)
display(threshold_df)
print("Selected threshold:")
display(pd.DataFrame([best_row]))

Best threshold: 0.675


,precision,recall,f1,f0_5,accuracy,threshold
0,0.833002,0.856851,0.844758,0.837665,0.993235,0.400
1,0.833002,0.856851,0.844758,0.837665,0.993235,0.425
2,0.833002,0.856851,0.844758,0.837665,0.993235,0.450
3,0.833002,0.856851,0.844758,0.837665,0.993263,0.475
4,0.834661,0.856851,0.845610,0.839007,0.993319,0.500
5,0.838000,0.856851,0.847321,0.841703,0.993430,0.525
6,0.837349,0.852761,0.844985,0.840387,0.993374,0.550
7,0.845842,0.852761,0.849287,0.847217,0.993486,0.575
8,0.838384,0.848671,0.843496,0.840421,0.993374,0.600
9,0.842105,0.850716,0.846389,0.843813,0.993486,0.625


Selected threshold:


,precision,recall,f1,f0_5,accuracy,threshold
11,0.861284,0.850716,0.855967,0.859149,0.993931,0.675


In [19]:
# ============================================================
# 16C. Evaluación test con threshold calibrado
# ============================================================

test_pred_output = trainer.predict(tokenized_dataset["test"])
test_logits = test_pred_output.predictions
test_labels = test_pred_output.label_ids

test_adjusted_pred_ids = apply_entity_threshold(
    test_logits,
    test_labels,
    threshold=BEST_ENTITY_THRESHOLD,
)

test_threshold_metrics = compute_seqeval_from_pred_ids(
    test_adjusted_pred_ids,
    test_labels,
)

validation_adjusted_pred_ids = apply_entity_threshold(
    val_logits,
    val_labels,
    threshold=BEST_ENTITY_THRESHOLD,
)

validation_threshold_metrics = compute_seqeval_from_pred_ids(
    validation_adjusted_pred_ids,
    val_labels,
)

threshold_metrics_report = {
    "best_entity_threshold": BEST_ENTITY_THRESHOLD,
    "validation_threshold_metrics": validation_threshold_metrics,
    "test_threshold_metrics": test_threshold_metrics,
}

with (REPORTS_DIR / "ner_threshold_metrics_v1_2.json").open("w", encoding="utf-8") as f:
    json.dump(threshold_metrics_report, f, indent=2, ensure_ascii=False)

print(json.dumps(threshold_metrics_report, indent=2, ensure_ascii=False))

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


{
  "best_entity_threshold": 0.675,
  "validation_threshold_metrics": {
    "precision": 0.8612836438923396,
    "recall": 0.8507157464212679,
    "f1": 0.8559670781893005,
    "f0_5": 0.859149111937216,
    "accuracy": 0.9939311266390134
  },
  "test_threshold_metrics": {
    "precision": 0.8100436681222707,
    "recall": 0.8100436681222707,
    "f1": 0.8100436681222707,
    "f0_5": 0.8100436681222707,
    "accuracy": 0.9931877435451768
  }
}


In [20]:
# ============================================================
# 16D. Evaluación separada por label_source y annotation_status
# ============================================================

def subset_tokenized_by_indices(split_name: str, indices: List[int]) -> Dataset:
    return tokenized_dataset[split_name].select(indices)


def evaluate_subset(split_df: pd.DataFrame, split_name: str, group_col: str) -> pd.DataFrame:
    rows = []

    for value, group in split_df.groupby(group_col, dropna=False):
        indices = group.index.tolist()
        if len(indices) < 5:
            continue

        subset = subset_tokenized_by_indices(split_name, indices)
        pred_output = trainer.predict(subset)
        logits = pred_output.predictions
        labels = pred_output.label_ids

        base_pred_ids = np.argmax(logits, axis=-1)
        base_metrics = compute_seqeval_from_pred_ids(base_pred_ids, labels)

        threshold_pred_ids = apply_entity_threshold(
            logits,
            labels,
            threshold=BEST_ENTITY_THRESHOLD,
        )
        threshold_metrics = compute_seqeval_from_pred_ids(threshold_pred_ids, labels)

        rows.append({
            "split": split_name,
            "group_col": group_col,
            "group_value": str(value),
            "rows": len(indices),
            "entities": int(group["entity_count"].sum()),
            "base_precision": base_metrics["precision"],
            "base_recall": base_metrics["recall"],
            "base_f1": base_metrics["f1"],
            "base_f0_5": base_metrics["f0_5"],
            "threshold_precision": threshold_metrics["precision"],
            "threshold_recall": threshold_metrics["recall"],
            "threshold_f1": threshold_metrics["f1"],
            "threshold_f0_5": threshold_metrics["f0_5"],
        })

    return pd.DataFrame(rows)


# Necesitamos reset_index para que los índices coincidan con tokenized_dataset.
val_eval_df = val_df.reset_index(drop=True)
test_eval_df = test_df.reset_index(drop=True)

eval_by_source_val = evaluate_subset(val_eval_df, "validation", "label_source")
eval_by_source_test = evaluate_subset(test_eval_df, "test", "label_source")
eval_by_status_val = evaluate_subset(val_eval_df, "validation", "annotation_status")
eval_by_status_test = evaluate_subset(test_eval_df, "test", "annotation_status")

eval_groups_df = pd.concat(
    [eval_by_source_val, eval_by_source_test, eval_by_status_val, eval_by_status_test],
    ignore_index=True,
)

eval_groups_path = REPORTS_DIR / "ner_eval_by_group_v1_2.csv"
eval_groups_df.to_csv(eval_groups_path, index=False, encoding="utf-8")

display(eval_groups_df)

/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


/usr/local/lib/python3.12/dist-packages/torch/autograd/function.py:583: UserWarning: Was asked to gather along dimension 0, but all input tensors were scalars; will instead unsqueeze and return a vector.
  return super().apply(*args, **kwargs)  # type: ignore[misc]


/usr/local/lib/python3.12/dist-packages/numpy/lib/_function_base_impl.py:557: RuntimeWarning: Mean of empty slice.
  avg = a.mean(axis, **keepdims_kw)
/usr/local/lib/python3.12/dist-packages/numpy/_core/_methods.py:138: RuntimeWarning: invalid value encountered in scalar divide
  ret = ret.dtype.type(ret / rcount)


,split,group_col,group_value,rows,entities,base_precision,base_recall,base_f1,base_f0_5,threshold_precision,threshold_recall,threshold_f1,threshold_f0_5
0,validation,label_source,manual_gold,102,158,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,validation,label_source,negative_or_unlabeled,266,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
2,validation,label_source,weak_hybrid,145,331,0.774481,0.788520,0.781437,0.777248,0.806250,0.779456,0.792627,0.800745
3,test,label_source,manual_gold,102,155,0.993548,0.993548,0.993548,0.993548,0.993506,0.987097,0.990291,0.992218
4,test,label_source,negative_or_unlabeled,266,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
5,test,label_source,weak_hybrid,145,303,0.716561,0.742574,0.729335,0.721616,0.726667,0.719472,0.723051,0.725216
6,validation,annotation_status,manual_augmented_multi_dish,49,114,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
7,validation,annotation_status,manual_augmented_negative_control,10,0,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
8,validation,annotation_status,manual_augmented_single_dish,42,42,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
9,validation,annotation_status,pending,411,331,0.756522,0.788520,0.772189,0.762712,0.793846,0.779456,0.786585,0.790926


In [21]:
# ============================================================
# 17. Predicción, threshold y postprocesado de spans
# ============================================================

LEADING_WORDS_TO_STRIP = {
    "el", "la", "los", "las",
    "un", "una", "unos", "unas",
    "del", "de", "al",
    "su", "sus", "mi", "mis",
}

TRAILING_WORDS_TO_STRIP = {
    "de", "del", "con", "y", "o", "u", "en", "al", "a", "para", "por", "que",
    "muy", "más", "menos",
}

BAD_SPAN_EXACT = {
    "comida", "cena", "almuerzo", "desayuno", "servicio", "ambiente",
    "precio", "local", "restaurante", "bar", "sitio", "mesa", "carta",
    "menú", "menu", "plato", "platos", "tapas", "tapa",
}

def clean_predicted_span(text: str, start: int, end: int) -> Optional[Dict[str, Any]]:
    original_start, original_end = start, end

    while start < end and text[start].isspace():
        start += 1
    while end > start and text[end - 1].isspace():
        end -= 1

    while start < end and text[start] in ".,;:¡!¿?()[]{}\"'“”‘’":
        start += 1
    while end > start and text[end - 1] in ".,;:¡!¿?()[]{}\"'“”‘’":
        end -= 1

    if end <= start:
        return None

    span_text = text[start:end].strip()
    words = span_text.split()

    while words and words[0].lower().strip(".,;:") in LEADING_WORDS_TO_STRIP:
        removed = words.pop(0)
        start += len(removed)
        while start < end and text[start].isspace():
            start += 1

    while words and words[-1].lower().strip(".,;:") in TRAILING_WORDS_TO_STRIP:
        removed = words.pop()
        end -= len(removed)
        while end > start and text[end - 1].isspace():
            end -= 1

    if end <= start:
        return None

    cleaned_text = text[start:end].strip()
    cleaned_norm = cleaned_text.lower()

    if not cleaned_text:
        return None

    if cleaned_norm in BAD_SPAN_EXACT:
        return None

    if len(cleaned_text.split()) > 8:
        return None

    return {
        "start": start,
        "end": end,
        "text": cleaned_text,
        "label": "DISH",
        "original_start": original_start,
        "original_end": original_end,
        "original_text": text[original_start:original_end],
    }


def bio_to_spans_with_threshold(
    text: str,
    offsets: List[Tuple[int, int]],
    pred_ids: List[int],
) -> List[Dict[str, Any]]:
    spans = []
    current_start = None
    current_end = None

    for offset, label_id in zip(offsets, pred_ids):
        start, end = offset

        if start == end:
            continue

        label = ID2LABEL[int(label_id)]

        if label == "B-DISH":
            if current_start is not None:
                cleaned = clean_predicted_span(text, current_start, current_end)
                if cleaned is not None:
                    spans.append(cleaned)
            current_start = start
            current_end = end

        elif label == "I-DISH" and current_start is not None:
            current_end = end

        else:
            if current_start is not None:
                cleaned = clean_predicted_span(text, current_start, current_end)
                if cleaned is not None:
                    spans.append(cleaned)
                current_start = None
                current_end = None

    if current_start is not None:
        cleaned = clean_predicted_span(text, current_start, current_end)
        if cleaned is not None:
            spans.append(cleaned)

    unique = []
    seen = set()
    for span in spans:
        key = (span["start"], span["end"], span["text"].lower())
        if key not in seen:
            unique.append(span)
            seen.add(key)

    return unique


def predict_entities(text: str, threshold: Optional[float] = None) -> List[Dict[str, Any]]:
    if threshold is None:
        threshold = BEST_ENTITY_THRESHOLD

    encoded = tokenizer(
        text,
        return_offsets_mapping=True,
        truncation=True,
        max_length=MAX_LENGTH,
        return_tensors="pt",
    )

    offsets = encoded.pop("offset_mapping")[0].tolist()
    encoded = {k: v.to(model.device) for k, v in encoded.items()}

    model.eval()
    with torch.no_grad():
        outputs = model(**encoded)
        logits = outputs.logits[0].detach().cpu().numpy()
        probs = softmax_np(logits)
        pred_ids = np.argmax(probs, axis=-1)

        for idx in range(len(pred_ids)):
            pred_id = int(pred_ids[idx])
            if pred_id in [LABEL2ID["B-DISH"], LABEL2ID["I-DISH"]]:
                if float(probs[idx, pred_id]) < threshold:
                    pred_ids[idx] = LABEL2ID["O"]

    return bio_to_spans_with_threshold(text, offsets, pred_ids.tolist())


examples = [
    "Las croquetas de rabo de toro estaban increíbles y el solomillo al whisky espectacular.",
    "La pizza estaba buena, pero la tarta de queso fue lo mejor de la noche.",
    "El servicio fue excelente y el ambiente muy agradable.",
    "Pedimos ensaladilla de gambas, salmorejo y un montadito de pringá.",
    "La tortilla de huevos con jamón estaba bien, pero el arroz salió frío.",
    "No probamos la carrillada, pero las croquetas estaban muy buenas.",
    "El pollito me gusta mucho, me encanta que me lo den crudo, y el cerdo carbonizado perfecto, que cerdos los camareros",
]

for text in examples:
    print("\nTEXT:", text)
    print("PRED:", predict_entities(text))


TEXT: Las croquetas de rabo de toro estaban increíbles y el solomillo al whisky espectacular.
PRED: [{'start': 4, 'end': 29, 'text': 'croquetas de rabo de toro', 'label': 'DISH', 'original_start': 4, 'original_end': 29, 'original_text': 'croquetas de rabo de toro'}, {'start': 54, 'end': 73, 'text': 'solomillo al whisky', 'label': 'DISH', 'original_start': 54, 'original_end': 73, 'original_text': 'solomillo al whisky'}]

TEXT: La pizza estaba buena, pero la tarta de queso fue lo mejor de la noche.
PRED: [{'start': 3, 'end': 8, 'text': 'pizza', 'label': 'DISH', 'original_start': 3, 'original_end': 8, 'original_text': 'pizza'}, {'start': 31, 'end': 45, 'text': 'tarta de queso', 'label': 'DISH', 'original_start': 31, 'original_end': 45, 'original_text': 'tarta de queso'}]

TEXT: El servicio fue excelente y el ambiente muy agradable.
PRED: []

TEXT: Pedimos ensaladilla de gambas, salmorejo y un montadito de pringá.
PRED: [{'start': 8, 'end': 19, 'text': 'ensaladilla', 'label': 'DISH', 'ori

In [22]:
# ============================================================
# 18. Predicciones de muestra y análisis de errores calibrado
# ============================================================

def gold_entities_to_simple(entities: List[Dict[str, Any]]) -> List[Dict[str, Any]]:
    return [
        {
            "start": int(e["start"]),
            "end": int(e["end"]),
            "text": e.get("text", ""),
            "label": "DISH",
        }
        for e in entities
    ]


sample_rows = []
error_rows = []

test_sample = test_df.copy().head(300)

for _, row in test_sample.iterrows():
    text = row["text"]
    gold = gold_entities_to_simple(row["entities"])
    pred = predict_entities(text, threshold=BEST_ENTITY_THRESHOLD)

    sample_rows.append({
        "document_id": row["document_id"],
        "review_id": row["review_id"],
        "annotation_status": row["annotation_status"],
        "label_source": row["label_source"],
        "text": text,
        "gold_entities_json": json.dumps(gold, ensure_ascii=False),
        "pred_entities_json": json.dumps(pred, ensure_ascii=False),
        "gold_count": len(gold),
        "pred_count": len(pred),
    })

    gold_set = {g["text"].lower().strip() for g in gold}
    pred_set = {p["text"].lower().strip() for p in pred}

    false_positives = pred_set - gold_set
    false_negatives = gold_set - pred_set

    if false_positives or false_negatives:
        error_rows.append({
            "document_id": row["document_id"],
            "review_id": row["review_id"],
            "annotation_status": row["annotation_status"],
            "label_source": row["label_source"],
            "text": text,
            "gold_entities_json": json.dumps(gold, ensure_ascii=False),
            "pred_entities_json": json.dumps(pred, ensure_ascii=False),
            "false_positive_count": len(false_positives),
            "false_negative_count": len(false_negatives),
            "false_positives": json.dumps(list(false_positives), ensure_ascii=False),
            "false_negatives": json.dumps(list(false_negatives), ensure_ascii=False),
        })

pred_sample_df = pd.DataFrame(sample_rows)
error_analysis_df = pd.DataFrame(error_rows)

pred_sample_path = PREDICTIONS_DIR / "ner_predictions_sample_v1_2.csv"
error_analysis_path = PREDICTIONS_DIR / "ner_error_analysis_v1_2.csv"

pred_sample_df.to_csv(pred_sample_path, index=False, encoding="utf-8")
error_analysis_df.to_csv(error_analysis_path, index=False, encoding="utf-8")

print("Pred sample:", pred_sample_path, pred_sample_df.shape)
print("Error analysis:", error_analysis_path, error_analysis_df.shape)
print("Best threshold:", BEST_ENTITY_THRESHOLD)

display(pred_sample_df.head(20))
display(error_analysis_df.head(20))

Pred sample: /kaggle/working/predictions/ner_predictions_sample_v1_2.csv (300, 9)
Error analysis: /kaggle/working/predictions/ner_error_analysis_v1_2.csv (38, 11)
Best threshold: 0.675


,document_id,review_id,annotation_status,label_source,text,gold_entities_json,pred_entities_json,gold_count,pred_count
0,sevilla_google_review_7eb51fc5-cdc9-4eda-a68f-...,7eb51fc5-cdc9-4eda-a68f-d555429c26b2,pending,negative_or_unlabeled,"Bar mediocre, la atención deja bastante que de...",[],[],0,0
1,sevilla_ner_manual_aug_0311,manual-ner-aug-bbc62213efafaa8ec1bf1fa9,manual_augmented_single_dish,manual_gold,Probamos vieiras gratinadas y nos pareció una ...,"[{""start"": 9, ""end"": 27, ""text"": ""vieiras grat...","[{""start"": 9, ""end"": 27, ""text"": ""vieiras grat...",1,1
2,sevilla_google_review_6a2bc793-b3fa-4fcd-b8ab-...,6a2bc793-b3fa-4fcd-b8ab-5f10ad56446f,pending,negative_or_unlabeled,"La cervecería está siempre hasta arriba, si va...",[],[],0,0
3,sevilla_google_review_85497db7-ca5e-4145-879f-...,85497db7-ca5e-4145-879f-a2785f41627f,pending,weak_hybrid,No suelen ser mis sitios predilectos este tipo...,"[{""start"": 205, ""end"": 210, ""text"": ""pizza"", ""...","[{""start"": 205, ""end"": 210, ""text"": ""pizza"", ""...",1,1
4,sevilla_google_review_61cd7d2a-0f1b-4e8b-a0da-...,61cd7d2a-0f1b-4e8b-a0da-a24b7b9a5ed5,pending,weak_hybrid,"He ido en un par de ocasiones a desayunar, y t...","[{""start"": 125, ""end"": 132, ""text"": ""churros"",...","[{""start"": 125, ""end"": 132, ""text"": ""churros"",...",1,1
5,sevilla_ner_manual_aug_0879,manual-ner-aug-1fd3568f5042223e6acb34b4,manual_augmented_multi_dish,manual_gold,"La mesa pidió coxinha brasileña, garbanzos con...","[{""start"": 14, ""end"": 31, ""text"": ""coxinha bra...","[{""start"": 14, ""end"": 31, ""text"": ""coxinha bra...",3,3
6,sevilla_google_review_e7bb190d-aea0-4315-9792-...,e7bb190d-aea0-4315-9792-5fba53fc78e4,pending,negative_or_unlabeled,Ambiente muy cuidado y muy buena atención. La ...,[],[],0,0
7,sevilla_google_review_1b5ccc05-58b1-4289-bb80-...,1b5ccc05-58b1-4289-bb80-6fca5dee87df,pending,negative_or_unlabeled,Puedo entender muchas cosas en el mundo de la ...,[],[],0,0
8,sevilla_google_review_e9f74ee5-4147-48be-979f-...,e9f74ee5-4147-48be-979f-5eab71f71f74,pending,weak_hybrid,"De los 3 días que estuvimos en Sevilla, dos de...","[{""start"": 218, ""end"": 226, ""text"": ""tortilla""...","[{""start"": 218, ""end"": 226, ""text"": ""tortilla""...",1,1
9,sevilla_google_review_da9481c3-d6dc-46bd-a577-...,da9481c3-d6dc-46bd-a577-47e0b2664d21,pending,weak_hybrid,Me ha parecido el sitio donde se desayuna mejo...,"[{""start"": 89, ""end"": 95, ""text"": ""pringá"", ""l...","[{""start"": 89, ""end"": 95, ""text"": ""pringá"", ""l...",2,1


,document_id,review_id,annotation_status,label_source,text,gold_entities_json,pred_entities_json,false_positive_count,false_negative_count,false_positives,false_negatives
0,sevilla_google_review_da9481c3-d6dc-46bd-a577-...,da9481c3-d6dc-46bd-a577-47e0b2664d21,pending,weak_hybrid,Me ha parecido el sitio donde se desayuna mejo...,"[{""start"": 89, ""end"": 95, ""text"": ""pringá"", ""l...","[{""start"": 89, ""end"": 95, ""text"": ""pringá"", ""l...",0,1,[],"[""secreto""]"
1,sevilla_google_review_fe5e6b9b-2ee3-4633-92a0-...,fe5e6b9b-2ee3-4633-92a0-42ddd8413b90,pending,weak_hybrid,Buena Calidad de la comida. El servicio ha est...,"[{""start"": 691, ""end"": 711, ""text"": "" croqueta...","[{""start"": 692, ""end"": 712, ""text"": ""croquetas...",1,3,"[""croquetas de puchero""]","[""bacala"", ""croquetas de pucher"", ""tosta de qu..."
2,sevilla_google_review_2b728b6b-cf5d-4e76-ae72-...,2b728b6b-cf5d-4e76-ae72-50bf29826c43,pending,negative_or_unlabeled,Nuestro favorito hasta el momento. Fuimos a pr...,[],"[{""start"": 207, ""end"": 217, ""text"": ""langostin...",1,0,"[""langostino""]",[]
3,sevilla_google_review_e2be016c-0da5-4d60-afd7-...,e2be016c-0da5-4d60-afd7-e6c5497fdcd8,pending,weak_hybrid,"Cafetería familiar, amplia y con terraza, lo c...","[{""start"": 410, ""end"": 417, ""text"": ""helados"",...","[{""start"": 410, ""end"": 416, ""text"": ""helado"", ...",1,1,"[""helado""]","[""helados""]"
4,sevilla_google_review_656fefb1-9ba1-4a80-bbd9-...,656fefb1-9ba1-4a80-bbd9-3a7095c132ed,pending,weak_hybrid,Ayer nos pasamos por uno de nuestros rincones ...,"[{""start"": 517, ""end"": 528, ""text"": "" y langos...","[{""start"": 520, ""end"": 531, ""text"": ""langostin...",2,2,"[""langostinos"", ""albóndigas""]","[""y langosti"", ""as albóndi""]"
5,sevilla_google_review_15e07008-5593-4f78-9199-...,15e07008-5593-4f78-9199-2c956e082794,pending,weak_hybrid,"Auténtica pringá casera, los churros inmejorab...","[{""start"": 10, ""end"": 16, ""text"": ""pringá"", ""l...","[{""start"": 10, ""end"": 21, ""text"": ""pringá case...",1,1,"[""pringá case""]","[""pringá""]"
6,sevilla_google_review_6e111548-f35a-4b7c-abf4-...,6e111548-f35a-4b7c-abf4-9f4800f91ef2,pending,weak_hybrid,Fui con una amiga hace unos días y nos encantó...,"[{""start"": 268, ""end"": 272, ""text"": ""açai"", ""l...",[],0,1,[],"[""açai""]"
7,sevilla_google_review_970a6478-7fa4-4947-95f5-...,970a6478-7fa4-4947-95f5-0d21defedafc,pending,weak_hybrid,De estos bares que te encuentras sin buscar y ...,"[{""start"": 303, ""end"": 313, ""text"": ""s montadi...","[{""start"": 305, ""end"": 315, ""text"": ""montadito...",3,4,"[""montaditos"", ""montadito"", ""patatas fritas""]","[""e montadit"", ""s montadit"", ""s y monta"", ""s p..."
8,sevilla_google_review_541b2d6b-7f39-4ce8-bfc3-...,541b2d6b-7f39-4ce8-bfc3-2f66ed18f7d5,pending,weak_hybrid,No tienen carta solo el incómodo código QR.\nO...,"[{""start"": 326, ""end"": 332, ""text"": "" choco"", ...","[{""start"": 327, ""end"": 333, ""text"": ""chocos"", ...",1,1,"[""chocos""]","[""choco""]"
9,sevilla_google_review_d1df4cb4-3745-45ee-87f3-...,d1df4cb4-3745-45ee-87f3-39857d2a2a0c,pending,weak_hybrid,"Hicimos reserva en el patio, éramos un grupo d...","[{""start"": 85, ""end"": 95, ""text"": "" albóndiga""...","[{""start"": 86, ""end"": 96, ""text"": ""albóndigas""...",5,5,"[""flamenquin"", ""albóndigas"", ""montadito"", ""pat...","[""patatas brava"", ""croqueta"", ""albóndiga"", ""fl..."


In [23]:
# ============================================================
# 19. Guardado de label mapping y resumen final
# ============================================================

label_mapping = {
    "label_list": LABEL_LIST,
    "label2id": LABEL2ID,
    "id2label": ID2LABEL,
}

with (MODEL_OUTPUT_DIR / "label_mapping.json").open("w", encoding="utf-8") as f:
    json.dump(label_mapping, f, indent=2, ensure_ascii=False)

final_summary = {
    "notebook": NOTEBOOK_NAME,
    "version": VERSION,
    "generated_at": datetime.now(timezone.utc).isoformat(),
    "model_checkpoint": MODEL_CHECKPOINT,
    "dataset_path": str(DATA_PATH),
    "dataset_rows": int(len(df)),
    "total_entities": int(df["entity_count"].sum()),
    "split_summary": split_summary,
    "training": train_metrics,
    "base_metrics": {
        "validation_metrics": validation_metrics,
        "test_metrics": test_metrics,
    },
    "threshold_calibration": {
        "best_entity_threshold": BEST_ENTITY_THRESHOLD,
        "validation_threshold_metrics": validation_threshold_metrics,
        "test_threshold_metrics": test_threshold_metrics,
    },
    "eval_by_group_csv": str(eval_groups_path),
    "label_mapping": label_mapping,
    "outputs": {
        "model_dir": str(MODEL_OUTPUT_DIR),
        "base_metrics_json": str(REPORTS_DIR / "ner_metrics_v1_2_base.json"),
        "threshold_metrics_json": str(REPORTS_DIR / "ner_threshold_metrics_v1_2.json"),
        "threshold_calibration_csv": str(REPORTS_DIR / "ner_threshold_calibration_v1_2.csv"),
        "eval_by_group_csv": str(eval_groups_path),
        "predictions_sample_csv": str(pred_sample_path),
        "error_analysis_csv": str(error_analysis_path),
    },
    "notes": [
        "NER v1.2 uses single-stage training, conservative class weights and label smoothing.",
        "The best checkpoint is selected using F0.5 to favor precision.",
        "A calibrated entity threshold is selected on validation.",
        "Inference includes span postprocessing to remove leading articles and trailing connectors.",
        "Evaluation is still partially based on weak labels, so a real manually reviewed test set is recommended.",
    ],
}

with (REPORTS_DIR / "ner_training_summary_v1_2.json").open("w", encoding="utf-8") as f:
    json.dump(final_summary, f, indent=2, ensure_ascii=False)

print(json.dumps(final_summary, indent=2, ensure_ascii=False))

{
  "notebook": "18_train_sevilla_dish_ner_model_v1_2",
  "version": "sevilla_dish_ner_beto_v1_2",
  "generated_at": "2026-05-12T16:54:16.424437+00:00",
  "model_checkpoint": "dccuchile/bert-base-spanish-wwm-cased",
  "dataset_path": "/kaggle/input/datasets/ivanarteagacordero/hidden-gems-sevilla-v1/sevilla_ner_annotation_dataset_v1_extended.csv",
  "dataset_rows": 5130,
  "total_entities": 4554,
  "split_summary": {
    "train_rows": 4104,
    "validation_rows": 513,
    "test_rows": 513,
    "train_entities": 3607,
    "validation_entities": 489,
    "test_entities": 458
  },
  "training": {
    "train_runtime": 445.8369,
    "train_samples_per_second": 55.231,
    "train_steps_per_second": 1.736,
    "total_flos": 1811978741426688.0,
    "train_loss": 0.321223588876946,
    "epoch": 4.0
  },
  "base_metrics": {
    "validation_metrics": {
      "eval_loss": 0.11273575574159622,
      "eval_precision": 0.8330019880715706,
      "eval_recall": 0.8568507157464212,
      "eval_f1": 0.844

In [24]:
# ============================================================
# 20. Crear ZIP descargable del modelo v1.2
# ============================================================

zip_path = OUTPUT_ROOT / f"{VERSION}.zip"

if zip_path.exists():
    zip_path.unlink()

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zipf:
    for file_path in MODEL_OUTPUT_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(MODEL_OUTPUT_DIR.parent))

    for file_path in REPORTS_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(OUTPUT_ROOT))

    for file_path in PREDICTIONS_DIR.rglob("*"):
        if file_path.is_file():
            zipf.write(file_path, file_path.relative_to(OUTPUT_ROOT))

print("ZIP creado:", zip_path)
print("Tamaño MB:", round(zip_path.stat().st_size / (1024 * 1024), 2))

ZIP creado: /kaggle/working/sevilla_dish_ner_beto_v1_2.zip
Tamaño MB: 2478.93
